<a href="https://colab.research.google.com/github/gusti-amber/rag/blob/main/chapter2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 12.9 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.23.0
    Uninstalling openai-2.23.0:
      Successfully uninstalled openai-2.23.0


In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "こんにちは！私はジョンと言います！"},
    ],
)
print(response.to_json(indent=2))

{
  "id": "chatcmpl-DDhJJobpUMLKRm3TyfsU2El9sI9lI",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "こんにちは、ジョンさん！お会いできて嬉しいです。今日はどんなことをお話ししたいですか？",
        "refusal": null,
        "role": "assistant",
        "annotations": []
      }
    }
  ],
  "created": 1772157121,
  "model": "gpt-4o-mini-2024-07-18",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": "fp_324beb61c9",
  "usage": {
    "completion_tokens": 28,
    "prompt_tokens": 25,
    "total_tokens": 53,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_tokens": 0,
      "cached_tokens": 0
    }
  }
}


In [ ]:
response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
      {"role": "system", "content": "You are a helpful assistant."},
      {"role": "user", "content": "こんにちは！私はジョンと言います！"},
  ],
  stream=True,
)

for chunk in response:
  content = chunk.choices[0].delta.content
  if content is not None:
      print(content, end="", flush=True)


こんにちは、ジョンさん！お会いできて嬉しいです。何かお手伝いできることがあれば教えてください！

In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
      {
          "role": "system",
          "content": '人物一覧を次のJSON形式で出力してください。\n{"people": ["aaa", "bbb"]}',
      },
      {
          "role": "user",
          "content": "昔々あるところにおじいさんとおばあさんがいました",
      },
  ],
  response_format={"type": "json_object"},
)
print(response.choices[0].message.content)

{"people": ["おじいさん", "おばあさん"]}


In [ ]:
import json

def get_current_weather(location, unit="fahrenheit"):
  if "tokyo" in location.lower():
      return json.dumps({"location": "Tokyo", "temperature": "10", "unit": unit})
  elif "san francisco" in location.lower():
      return json.dumps({"location": "San Francisco", "temperature": "72", "unit": unit})
  elif "paris" in location.lower():
      return json.dumps({"location": "Paris", "temperature": "22", "unit": unit})
  else:
      return json.dumps({"location": location, "temperature": "unknown"})


In [ ]:
tools = [
  {
      "type": "function",
      "function": {
          "name": "get_current_weather",
          "description": "Get the current weather in a given location",
          "parameters": {
              "type": "object",
              "properties": {
                  "location": {
                      "type": "string",
                      "description": "The city and state, e.g. San Francisco, CA",
                  },
                  "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
              },
              "required": ["location"],
          },
      },
  }
]

In [ ]:
from openai import OpenAI

client = OpenAI()

messages = [
  {"role": "user", "content": "東京の天気はどうですか？"},
]

response = client.chat.completions.create(
  model="gpt-4o",
  messages=messages,
  tools=tools,
)
print(response.to_json(indent=2))

{
  "id": "chatcmpl-DDiFOIuxm3oj909GW5YGhmycylPPe",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "tool_calls": [
          {
            "id": "call_Dxg0JOqyz5GJH5QDCcOUI9Vy",
            "function": {
              "arguments": "{\"location\":\"Tokyo\"}",
              "name": "get_current_weather"
            },
            "type": "function"
          }
        ]
      }
    }
  ],
  "created": 1772160722,
  "model": "gpt-4o-2024-08-06",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": "fp_6771f00d8a",
  "usage": {
    "completion_tokens": 15,
    "prompt_tokens": 81,
    "total_tokens": 96,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
  

In [ ]:
response_message = response.choices[0].message
messages.append(response_message.to_dict())

In [ ]:
available_functions = {
  "get_current_weather": get_current_weather,
}

# 使いたい関数は複数あるかもしれないのでループ
for tool_call in response_message.tool_calls:
  # 関数を実行
  function_name = tool_call.function.name
  function_to_call = available_functions[function_name]
  function_args = json.loads(tool_call.function.arguments)
  function_response = function_to_call(
      location=function_args.get("location"),
      unit=function_args.get("unit"),
  )

  # 関数の実行結果を会話履歴としてmessagesに追加
  messages.append(
      {
          "tool_call_id": tool_call.id,
          "role": "tool",
          "name": function_name,
          "content": function_response,
      }
  )

In [ ]:
print(json.dumps(messages, ensure_ascii=False, indent=2))

[
  {
    "role": "user",
    "content": "東京の天気はどうですか？"
  },
  {
    "content": null,
    "refusal": null,
    "role": "assistant",
    "annotations": [],
    "tool_calls": [
      {
        "id": "call_Dxg0JOqyz5GJH5QDCcOUI9Vy",
        "function": {
          "arguments": "{\"location\":\"Tokyo\"}",
          "name": "get_current_weather"
        },
        "type": "function"
      }
    ]
  },
  {
    "tool_call_id": "call_Dxg0JOqyz5GJH5QDCcOUI9Vy",
    "role": "tool",
    "name": "get_current_weather",
    "content": "{\"location\": \"Tokyo\", \"temperature\": \"10\", \"unit\": null}"
  }
]


In [ ]:
second_response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
)
print(second_response.to_json(indent=2))

{
  "id": "chatcmpl-DDiLTWXv81ZKDcDtuadGudX7eNADW",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "現在、東京の気温は10度です。",
        "refusal": null,
        "role": "assistant",
        "annotations": []
      }
    }
  ],
  "created": 1772161099,
  "model": "gpt-4o-2024-08-06",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": "fp_64dfa806c7",
  "usage": {
    "completion_tokens": 11,
    "prompt_tokens": 57,
    "total_tokens": 68,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_tokens": 0,
      "cached_tokens": 0
    }
  }
}
